# 04 — Explicabilité SHAP : `contains_sulfates`

Cible choisie car c'est la seule pour laquelle le modèle image (CLIP) apporte un vrai gain par rapport à la catégorie seule (AUC 0.725 vs 0.716, et 0.743 combiné).

Deux analyses complémentaires :
1. **SHAP sur les features manuelles** (couleur/contraste/luminosité) — directement interprétable, dit *quel attribut visuel simple* compte.
2. **Importance des catégories dans le modèle CLIP+catégorie** — dit si le signal vient surtout de la catégorie ou des embeddings.

(Les embeddings CLIP bruts (512 dimensions abstraites) ne sont pas interprétables directement avec SHAP — on ne peut pas dire ce que "la dimension 47" représente visuellement. On se concentre donc sur ce qui est explicable.)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from PIL import Image
import os
import shap
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TARGET = "contains_sulfates"

df_meta = pd.read_parquet("../data/processed/dataset_clean.parquet")
df_manual = pd.read_parquet("../data/processed/features_manual.parquet")
embeddings = np.load("../data/processed/embeddings_clip.npy")
embedding_ids = np.load("../data/processed/embeddings_ids.npy")

df_emb = pd.DataFrame(embeddings, columns=[f"clip_{i}" for i in range(embeddings.shape[1])])
df_emb["id"] = embedding_ids

df_full = df_meta.merge(df_manual, on="id").merge(df_emb, on="id")
category_dummies = pd.get_dummies(df_full["category"], prefix="cat")
df_full = pd.concat([df_full, category_dummies], axis=1)
cat_cols = category_dummies.columns.tolist()

manual_cols = ["dominant_r", "dominant_g", "dominant_b", "second_r", "second_g", "second_b",
               "contrast", "brightness", "saturation", "aspect_ratio"]
clip_cols = [c for c in df_full.columns if c.startswith("clip_")]

y = df_full[TARGET].astype(int)
idx_train, idx_test = train_test_split(df_full.index, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
y_train, y_test = y.loc[idx_train], y.loc[idx_test]

print(f"Dataset prêt. Distribution : {y.value_counts(normalize=True).round(3).to_dict()}")

## 1. Modèle pour SHAP — features manuelles (Gradient Boosting)

In [ ]:
X_manual_train = df_full.loc[idx_train, manual_cols]
X_manual_test = df_full.loc[idx_test, manual_cols]

model_manual = GradientBoostingClassifier(random_state=RANDOM_STATE)
model_manual.fit(X_manual_train, y_train)

auc = roc_auc_score(y_test, model_manual.predict_proba(X_manual_test)[:, 1])
print(f"AUC-ROC (rappel) : {auc:.3f}")

## 2. SHAP summary plot — quels attributs visuels comptent ?

In [ ]:
explainer = shap.TreeExplainer(model_manual)
shap_values = explainer.shap_values(X_manual_test)

shap.summary_plot(shap_values, X_manual_test, show=False)
plt.title(f"SHAP summary — {TARGET}")
plt.tight_layout()
plt.show()

Lecture : chaque point est un produit. Rouge = valeur élevée de la feature, bleu = valeur faible. La position horizontale indique l'impact sur la prédiction (droite = pousse vers `contains_sulfates=True`).

## 3. SHAP bar plot — importance moyenne absolue

In [ ]:
shap.summary_plot(shap_values, X_manual_test, plot_type="bar", show=False)
plt.title(f"Importance moyenne des features — {TARGET}")
plt.tight_layout()
plt.show()

## 4. Importance de la catégorie dans le modèle CLIP + catégorie

Le modèle CLIP+catégorie est une régression logistique (pas d'arbre) — on regarde directement la magnitude des coefficients associés aux variables de catégorie, pour voir leur poids relatif face aux 512 dimensions CLIP.

In [ ]:
X_ctrl_train = df_full.loc[idx_train, clip_cols + cat_cols]
X_ctrl_test = df_full.loc[idx_test, clip_cols + cat_cols]

scaler = StandardScaler()
X_ctrl_train_s = scaler.fit_transform(X_ctrl_train)
X_ctrl_test_s = scaler.transform(X_ctrl_test)

model_ctrl = LogisticRegression(max_iter=2000, C=0.1, random_state=RANDOM_STATE)
model_ctrl.fit(X_ctrl_train_s, y_train)

auc_ctrl = roc_auc_score(y_test, model_ctrl.predict_proba(X_ctrl_test_s)[:, 1])
print(f"AUC-ROC (rappel, CLIP+catégorie) : {auc_ctrl:.3f}")

coefs = pd.Series(model_ctrl.coef_[0], index=clip_cols + cat_cols)

# Comparaison : magnitude moyenne des coefficients CLIP vs catégorie
mean_abs_clip = coefs[clip_cols].abs().mean()
mean_abs_cat = coefs[cat_cols].abs().mean()
print(f"\nMagnitude moyenne |coef| — dimensions CLIP : {mean_abs_clip:.4f}")
print(f"Magnitude moyenne |coef| — catégories : {mean_abs_cat:.4f}")

In [ ]:
cat_coefs = coefs[cat_cols].sort_values()
plt.figure(figsize=(8, 4))
plt.barh(cat_coefs.index.str.replace("cat_", ""), cat_coefs.values)
plt.axvline(0, color="black", linewidth=0.8)
plt.title(f"Coefficients par catégorie — {TARGET}")
plt.xlabel("Coefficient (positif = pousse vers True)")
plt.tight_layout()
plt.show()

## 5. Aperçu visuel — produits avec le plus fort score prédit

Contrôle qualitatif : à quoi ressemblent les produits que le modèle CLIP+catégorie considère comme les plus probables d'avoir des sulfates, et les moins probables ?

In [ ]:
proba_test = model_ctrl.predict_proba(X_ctrl_test_s)[:, 1]
df_test_view = df_full.loc[idx_test, ["brand", "name", "category", "image_path", TARGET]].copy()
df_test_view["proba_predite"] = proba_test

top_true = df_test_view.sort_values("proba_predite", ascending=False).head(5)
top_false = df_test_view.sort_values("proba_predite", ascending=True).head(5)

def show_row(subset, label):
    fig, axes = plt.subplots(1, len(subset), figsize=(3 * len(subset), 3.5))
    for ax, (_, row) in zip(axes, subset.iterrows()):
        img = Image.open(os.path.join("..", row["image_path"]))
        ax.imshow(img)
        ax.set_title(f"{row['brand'][:15]}\np={row['proba_predite']:.2f} | réel={row[TARGET]}", fontsize=8)
        ax.axis("off")
    plt.suptitle(label)
    plt.tight_layout()
    plt.show()

show_row(top_true, "Probabilité prédite la plus haute (sulfates)")
show_row(top_false, "Probabilité prédite la plus basse (sulfates)")

## 6. Synthèse pour le rapport

Points à documenter à partir de cette analyse :
- Quelles features visuelles simples (couleur, contraste, luminosité) ressortent le plus dans le SHAP summary plot, et dans quel sens (positif/négatif) ?
- Le poids relatif catégorie vs embeddings CLIP dans le modèle combiné — le signal visuel est-il concentré sur certaines catégories ou transversal ?
- Cohérence qualitative : les exemples à forte/faible probabilité prédite confirment-ils un pattern visuel reconnaissable, ou le modèle semble-t-il capter du bruit malgré l'AUC correcte ?